# Image-Level EDA
The bias matching process applied in the previous metadata EDA notebook tells us that the bias matching can be applied to reduce the bias in the dataset. However, it is also important to perform an image-level EDA to understand the distribution of images and their characteristics.

Crucially, this notebook combines multiple per source manifests in `data/interim` (produced by `scripts/build_manifest.py`) into a single combined manifest (`data/interim/manifest_final.parquet`) with paths pointing to the final, model-ready image folders under `data/processed/`, allowing for an overall image analysis. Train/val/test splits are allocated here.

The pipeline this notebook follows is as follows:


```text
per-source manifests (data/interim/*.parquet)
        |
        v
combine_manifests()          -- into one DataFrame standardized to canonical columns
        |
        v
drop is_corrupt rows          -- ignored going forward
        |
        v
assign_group_ids()             -- cross-source near-duplicate clustering (banded LSH)
        |
        v
apply_matching()                -- bias-match real GenImage images (size + JPEG QF)
        |
        v
assign_full_splits()           -- group-aware GenImage split + fixed lookup for
                                   coco / ntire / raise
        |
        v
assert_no_leakage()             -- hard stop if any group/hash spans two splits
        |
        v
save data/interim/manifest_final.parquet
        |
        v
build_processed_dataset()      -- resize + re-encode every surviving image
        |
        v
data/processed/<split>/<ai|human>/<sha256>.jpg
        +
data/processed/manifest.parquet

### 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

## Standard Libraries
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

## Project module imports
from ai_detector.data.selection import(
    SubsetConfig, combine_manifests, assign_group_ids, apply_matching, assign_full_splits, 
    normalize_columns, shortcut_probe)
from ai_detector.data.manifest import(assert_no_leakage, LeakageError)
from ai_detector.data import viz
from ai_detector.preprocessing.image_ops import PreprocessConfig, build_processed_dataset

## Path setup
PROJECT_ROOT = Path.cwd().parent
DATA = PROJECT_ROOT / "data"
INTERIM = DATA / "interim"
PROCESSED = DATA / "processed"
RAW = DATA / "raw"
CONFIG_PATH = PROJECT_ROOT / "configs" / "data" / "subset_v1.yaml"

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

## Preprocessing and data splitting will draw from a yaml file in configs/data
cfg = SubsetConfig.from_yaml(CONFIG_PATH)
raw_cfg = yaml.safe_load(CONFIG_PATH.read_text())
preprocess_cfg = PreprocessConfig.from_yaml_dict(raw_cfg.get("preprocessing", {}))

print(cfg)
print(preprocess_cfg)

SubsetConfig(name='subset_v1', seed=42, real_generator_token='nature', train_generators=['stable_diffusion_v_1_4', 'stable_diffusion_v_1_5', 'glide', 'adm', 'vqdm'], ood_generators=['midjourney', 'wukong', 'biggan'], min_side=450, max_side=550, jpeg_qf=98, jpeg_qf_tolerance=2, val_fraction=0.1, test_in_dist_fraction=0.1, pilot_n_per_stratum=60)
PreprocessConfig(image_size=512, jpeg_qf=98, resample=<Resampling.LANCZOS: 1>)


### 1. Load and combine per-source manifests
Draws from the per-source manifests stored in `data/interim`

In [3]:
manifest_paths = sorted(INTERIM.glob("manifest_*.parquet"))
print(f"Found {len(manifest_paths)} manifest files:")
for p in manifest_paths:
    print(" -", p.name)

df = combine_manifests(manifest_paths)
print(f"\nCombined manifest: {len(df):,} rows, {df['source'].nunique()} sources")
display(df.groupby(["source", "label"]).size().unstack(fill_value=0))


Found 4 manifest files:
 - manifest_coco.parquet
 - manifest_genimage_tiny.parquet
 - manifest_ntire.parquet
 - manifest_raise.parquet

Combined manifest: 90,300 rows, 4 sources


label,0,1
source,,
coco,5000,0
genimage,17500,17500
ntire,17982,32018
raise,300,0
